In [ ]:
%pip install psycopg2-binary --quiet

In [ ]:
import boto3, json
import psycopg2
import pandas as pd
from datetime import datetime, timezone

SERVICE_CREDENTIAL_NAME = "opex_prd_service_credential"
REGION = "eu-west-1"
SECRET_ID = "rds/opex/prd"

# ============================================================
# CONFIGURATION
# ============================================================
TARGET_CATALOG = "opex_gold"
TARGET_SCHEMA = "gold"

provider = dbutils.credentials.getServiceCredentialsProvider(SERVICE_CREDENTIAL_NAME)

session = boto3.Session(
    botocore_session=provider,
    region_name=REGION
)

sm = session.client("secretsmanager")
response = sm.get_secret_value(SecretId=SECRET_ID)

# Parse the secret - adjust keys based on actual secret structure
secret = json.loads(response["SecretString"]) if isinstance(response.get("SecretString"), str) else {"password": response["SecretString"]}
print("Secret keys available:", list(secret.keys()))


TABLE_PKS = {
    "access_plants": ['access_id', 'plant_id'],
    "accesses": ['id'],
    "alembic_version": ['version_num'],
    "answer_comments": ['id'],
    "answers": ['id'],
    "assessment_review_owners": ['assessment_id', 'user_id'],
    "assessment_selected_elements": ['assessment_id', 'element_number'],
    "assessment_technologies": ['assessment_id', 'technology'],
    "assessment_type_elements": ['assessment_type_id', 'element_number'],
    "assessment_types": ['id'],
    "assessment_viewers": ['assessment_id', 'user_id'],
    "assessments": ['id'],
    "audit_events": ['id'],
    "document_links": ['id'],
    "documents": ['id'],
    "element_owners": ['id'],
    "elements": ['element_number'],
    "entities": ['id'],
    "favorite_documents": ['id'],
    "frameworks": ['id'],
    "jobs": ['id'],
    "notifications": ['id'],
    "question_frameworks": ['question_id', 'framework'],
    "question_technologies": ['question_id', 'technology'],
    "questions": ['id'],
    "site_technologies": ['site_id', 'technology'],
    "sites": ['id'],
    "ticket_updates": ['id'],
    "tickets": ['id'],
}


# ============================================================
# CONNECTION
# ============================================================
def connect_pg():
    return psycopg2.connect(
        host="example.invalid",
        port=secret.get("DB_PORT", 5432),
        dbname=secret.get("DB_NAME", secret.get("dbname", "postgres")),
        user=secret.get("DB_USER", secret.get("username", "")),
        password=secret.get("DB_PASSWORD", secret.get("password", ""))
    )

# ============================================================
# SYNC: Full Refresh + MERGE (one strategy for all tables)
# ============================================================
def sync_table(conn, table_name, pk_cols):
    """Read entire source table, MERGE into Delta target.
    Handles inserts, updates, and deletes in one pass.
    Returns a dict with operation metrics."""
    target = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"

    # Full read from PostgreSQL
    pdf = pd.read_sql(f'SELECT * FROM public."{table_name}"', conn)

    if pdf.empty:
        print(f"  SKIP  {table_name} (0 rows in source)")
        return {"table": table_name, "inserted": 0, "updated": 0, "deleted": 0, "status": "skipped"}

    # First run: create table if it doesn't exist yet
    if not spark.catalog.tableExists(target):
        sdf = spark.createDataFrame(pdf)
        sdf.write.format("delta").mode("overwrite").saveAsTable(target)
        print(f"  CREATE {target} ({len(pdf)} rows) [initial load]")
        return {"table": table_name, "inserted": len(pdf), "updated": 0, "deleted": 0, "status": "created"}

    # Subsequent runs: MERGE (upsert + delete)
    sdf = spark.createDataFrame(pdf)
    sdf.createOrReplaceTempView(f"_src_{table_name}")

    merge_on = " AND ".join([f"target.`{pk}` = source.`{pk}`" for pk in pk_cols])
    non_pk_cols = [c for c in pdf.columns if c not in pk_cols]

    # Build MERGE statement
    insert_cols = ", ".join([f"`{c}`" for c in pdf.columns])
    insert_vals = ", ".join([f"source.`{c}`" for c in pdf.columns])

    if non_pk_cols:
        update_set = ", ".join([f"target.`{c}` = source.`{c}`" for c in non_pk_cols])
        merge_sql = f"""
            MERGE INTO {target} AS target
            USING _src_{table_name} AS source
            ON {merge_on}
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """
    else:
        # Junction tables with only PK columns (no fields to update)
        merge_sql = f"""
            MERGE INTO {target} AS target
            USING _src_{table_name} AS source
            ON {merge_on}
            WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """

    # Execute MERGE and capture metrics
    result = spark.sql(merge_sql)
    metrics = result.collect()[0]
    
    inserted = metrics["num_inserted_rows"]
    updated = metrics["num_updated_rows"]
    deleted = metrics["num_deleted_rows"]

    # Print with details
    changes = []
    if inserted > 0:
        changes.append(f"+{inserted} inserted")
    if updated > 0:
        changes.append(f"~{updated} updated")
    if deleted > 0:
        changes.append(f"-{deleted          } deleted")
    
    if changes:
        print(f"  MERGE {table_name:<30} {' | '.join(changes)}")
    else:
        print(f"  MERGE {table_name:<30} (no changes)")

    return {"table": table_name, "inserted": inserted, "updated": updated, "deleted": deleted, "status": "synced"}

# ============================================================
# RUN
# ============================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

run_start = datetime.now(timezone.utc)
print(f"{'='*70}")
print(f"FULL REFRESH + MERGE — {run_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"{'='*70}\n")

conn = connect_pg()
all_metrics = []
failures = []

for table_name, pk_cols in TABLE_PKS.items():
    try:
        metrics = sync_table(conn, table_name, pk_cols)
        all_metrics.append(metrics)
    except Exception as e:
        print(f"  FAIL  {table_name}: {e}")
        failures.append((table_name, str(e)))

conn.close()

# ============================================================
# SUMMARY REPORT
# ============================================================
run_end = datetime.now(timezone.utc)
total_inserted = sum(m["inserted"] for m in all_metrics)
total_updated = sum(m["updated"] for m in all_metrics)
total_deleted = sum(m["deleted"] for m in all_metrics)
tables_with_changes = [m for m in all_metrics if m["inserted"] + m["updated"] + m["deleted"] > 0]

print(f"\n{'='*70}")
print(f"SYNC SUMMARY")
print(f"{'='*70}")
print(f"  Duration:     {(run_end - run_start).total_seconds():.1f}s")
print(f"  Tables:       {len(TABLE_PKS)} total, {len(tables_with_changes)} with changes")
print(f"  Inserted:     {total_inserted} rows")
print(f"  Updated:      {total_updated} rows")
print(f"  Deleted:      {total_deleted} rows")
if failures:
    print(f"  Failures:     {len(failures)}")
    for name, err in failures:
        print(f"    - {name}: {err}")
print(f"{'='*70}")

# Display as a summary table
if tables_with_changes:
    print(f"\n\nTables with changes:")
    changes_df = pd.DataFrame(tables_with_changes)
    changes_df = changes_df[changes_df['status'] != 'skipped']
    changes_df = changes_df.sort_values(by=['inserted', 'updated', 'deleted'], ascending=False)
    display(changes_df)